# Soldani - Second task - Benchmark multi-dataset

Estende il benchmark FairMind vs LLM (vedi `2_3_benchmark_thor.ipynb`) a piu' dataset processati in `data/processed/`, invece del solo `adult.csv`.

## 1. Setup del path di progetto

Individua automaticamente la directory radice del progetto cercando la cartella src/ nella gerarchia superiore, quindi la aggiunge a sys.path per consentire gli import assoluti.

In [ ]:
from pathlib import Path
import sys

# Find the root by searching the "src" folder
current = Path.cwd()

while current != current.parent:
    if (current / "src").exists():
        REPO_ROOT = current
        break
    current = current.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


## 2. Import librerie e configurazione client LLM

Importa pandas, json, pgmpy e i moduli causali di FairMind (build_sfm, fit_discrete_bayesian_model, effetti). L'endpoint del server llama.cpp viene letto dalle variabili d'ambiente LLAMA_HOST/LLAMA_PORT (fallback localhost:8080), cosi' lo stesso notebook funziona sia in locale sia su THOR.

In [ ]:
import json
import os
import time
import datetime

import pandas as pd

from pgmpy.estimators import BayesianEstimator
from src.graph import build_sfm
from src.model import fit_discrete_bayesian_model
from src.effects import (
    total_variation, total_effect, spurious_effect,
    natural_direct_effect, natural_indirect_effect,
)
from src.llm import LLM_CONFIGS, call_llm

LLAMA_HOST = os.environ.get("LLAMA_HOST", "localhost")
LLAMA_PORT = os.environ.get("LLAMA_PORT", "8080")
LLM_CONFIGS[0]["base_url"] = f"http://{LLAMA_HOST}:{LLAMA_PORT}/v1"
print(f"LLM endpoint configurato: http://{LLAMA_HOST}:{LLAMA_PORT}/v1")


## 3. Configurazione dei benchmark — 10 dataset

Ogni dataset ha un proprio `CONFIG`: attributo protetto (`x0`=gruppo di riferimento per il confronto, `x1`=gruppo di interesse), target, mediatori e confounder. Le variabili continue usate come mediatori vengono discretizzate tramite la chiave opzionale `binning` (colonna -> bins/labels per `pd.cut`), in modo analogo a come `hours-per-week` era binnata nel notebook originale su `adult`.

**Nota**: la scelta di quali variabili siano mediatori/confounder per ciascun dataset e' un'ipotesi di modellazione causale ragionevole ma non definitiva — va rivista prima di trarre conclusioni scientifiche. Sono stati esclusi da questo giro `crimedata.csv` (attributo protetto continuo, non categorico), `compas-scores-two-years-violent.csv` (variante ridondante/molto sbilanciata di compas), `UCI_Credit_Card.csv` e `dutch_census_2001.csv` (attributi protetti codificati come interi 1/2 senza un dizionario di label affidabile nel dataset processato).

In [ ]:
CONFIGS = [
    {
        "dataset_name": "adult",
        "csv_path": "../../data/processed/adult.csv",
        "target_col": "T_income",
        "target_val": ">50K",
        "protected": "S2_gender",
        "x0": "Female",
        "x1": "Male",
        "mediators": ["hours-per-week"],
        "confounders": ["education"],
        "binning": {
            "hours-per-week": {
                "bins": [0, 20, 35, 45, 60, 100],
                "labels": ["<=20", "21-35", "36-45", "46-60", ">60"],
            },
        },
    },
    {
        "dataset_name": "census_income_kdd",
        "csv_path": "../../data/processed/Census_income_kdd.csv",
        "target_col": "T_income_level",
        "target_val": 1,
        "protected": "S1_sex",
        "x0": "Female",
        "x1": "Male",
        "mediators": ["weeks_worked_in_year"],
        "confounders": ["education"],
        "binning": {
            "weeks_worked_in_year": {
                "bins": [-1, 0, 13, 26, 39, 52],
                "labels": ["0", "1-13", "14-26", "27-39", "40-52"],
            },
        },
    },
    {
        "dataset_name": "bank_marketing",
        "csv_path": "../../data/processed/bank_marketing.csv",
        "target_col": "T_deposit",
        "target_val": "yes",
        "protected": "S2_marital",
        "x0": "single",
        "x1": "married",
        "mediators": ["housing"],
        "confounders": ["education"],
        "binning": {},
    },
    {
        "dataset_name": "compas_two_year_recid",
        "csv_path": "../../data/processed/compas-scores-two-years.csv",
        "target_col": "T_two_year_recid",
        "target_val": 1,
        "protected": "S1_race",
        "x0": "Caucasian",
        "x1": "African-American",
        "mediators": ["priors_count"],
        "confounders": ["age_cat"],
        "binning": {
            "priors_count": {
                "bins": [-1, 0, 2, 5, 10, 50],
                "labels": ["0", "1-2", "3-5", "6-10", ">10"],
            },
        },
    },
    {
        "dataset_name": "diabetes_130",
        "csv_path": "../../data/processed/diabetes_130.csv",
        "target_col": "T_readmitted",
        "target_val": "YES",
        "protected": "S1_gender",
        "x0": "Female",
        "x1": "Male",
        "mediators": ["time_in_hospital"],
        "confounders": ["age"],
        "binning": {
            "time_in_hospital": {
                "bins": [0, 3, 7, 14],
                "labels": ["1-3", "4-7", "8-14"],
            },
        },
    },
    {
        "dataset_name": "german_credit",
        "csv_path": "../../data/processed/german_credit_complete.csv",
        "target_col": "T_Creditability",
        "target_val": 1,
        "protected": "S2_Sex",
        "x0": "Female",
        "x1": "Male",
        "mediators": ["Length of current employment"],
        "confounders": ["Occupation"],
        "binning": {},
    },
    {
        "dataset_name": "law_bar_pass",
        "csv_path": "../../data/processed/law_bar_pass_prediction.csv",
        "target_col": "T_bar_passed",
        "target_val": True,
        "protected": "S2_race",
        "x0": "Other",
        "x1": "White",
        "mediators": ["ugpa"],
        "confounders": ["fam_inc"],
        "binning": {
            "ugpa": {
                "bins": [0, 2.5, 3.0, 3.5, 4.0],
                "labels": ["<2.5", "2.5-3.0", "3.0-3.5", "3.5-4.0"],
            },
        },
    },
    {
        "dataset_name": "oulad_final_result",
        "csv_path": "../../data/processed/studentInfo_OULAD.csv",
        "target_col": "T_final_result",
        "target_val": "Pass",
        "protected": "S1_gender",
        "x0": "F",
        "x1": "M",
        "mediators": ["studied_credits"],
        "confounders": ["highest_education"],
        "binning": {
            "studied_credits": {
                "bins": [0, 60, 120, 180, 240, 700],
                "labels": ["<=60", "61-120", "121-180", "181-240", ">240"],
            },
        },
    },
    {
        "dataset_name": "student_mat",
        "csv_path": "../../data/processed/student_mat.csv",
        "target_col": "T_grade",
        "target_val": 1,
        "protected": "S2_sex",
        "x0": "F",
        "x1": "M",
        "mediators": ["studytime"],
        "confounders": ["Medu"],
        "binning": {},
    },
    {
        "dataset_name": "student_por",
        "csv_path": "../../data/processed/student_por.csv",
        "target_col": "T_grade",
        "target_val": 1,
        "protected": "S2_sex",
        "x0": "F",
        "x1": "M",
        "mediators": ["studytime"],
        "confounders": ["Medu"],
        "binning": {},
    },
]

print(f"{len(CONFIGS)} dataset configurati: {[c['dataset_name'] for c in CONFIGS]}")


## 4. Funzioni generalizzate

`apply_binning` discretizza le colonne indicate in `config["binning"]` (generalizza la logica hardcoded su `hours-per-week` del notebook originale). `run_fairmind` e `build_llm_prompt` ora filtrano il dataset sulle sole righe con attributo protetto in `{x0, x1}` (necessario per dataset come `diabetes_130`, che ha una terza categoria `Unknown/Invalid`, o `compas`, che ha piu' di due razze) e applicano il binning in modo generico invece che su una singola colonna hardcoded.

In [ ]:
def apply_binning(df: pd.DataFrame, config: dict) -> tuple[pd.DataFrame, list[str], list[str]]:
    """Discretizza le colonne mediatore indicate in config["binning"] tramite pd.cut.

    Ritorna il df aggiornato, i nomi delle colonne mediatore da usare a valle
    (binnate o originali) e una lista di note testuali sul binning applicato
    (da inserire nel prompt LLM).
    """
    binning_spec = config.get("binning", {})
    binned_mediators = []
    notes = []
    for col in config["mediators"]:
        if col in binning_spec:
            spec = binning_spec[col]
            bin_col = f"{col}_bin"
            df[bin_col] = pd.cut(
                df[col],
                bins=spec["bins"],
                labels=spec["labels"],
                include_lowest=True,
            )
            binned_mediators.append(bin_col)
            notes.append(f'"{col}" discretized into bins: {", ".join(spec["labels"])}')
        else:
            binned_mediators.append(col)
    return df, binned_mediators, notes


def load_filtered(config: dict) -> pd.DataFrame:
    """Carica il CSV, seleziona le colonne rilevanti e filtra sulle sole
    categorie x0/x1 dell'attributo protetto."""
    cols = (
        [config["protected"]]
        + config["mediators"]
        + config["confounders"]
        + [config["target_col"]]
    )
    df = pd.read_csv(config["csv_path"])[cols].dropna()
    df = df[df[config["protected"]].isin([config["x0"], config["x1"]])]
    return df


In [ ]:
def run_fairmind(config: dict) -> tuple[dict, float]:
    df = load_filtered(config)
    df, mediators, _ = apply_binning(df, config)

    sfm = build_sfm(
        sensitive_attr=config["protected"],
        outcome_attr=config["target_col"],
        confounder_attrs=config["confounders"],
        mediator_attrs=mediators,
        sorted_mediators=len(mediators) > 1,
        sorted_confounders=len(config["confounders"]) > 1,
    )
    bn = fit_discrete_bayesian_model(
        sfm=sfm,
        data=df,
        estimator_instance=(BayesianEstimator, {"prior_type": "BDeu"}),
    )

    target = (config["target_col"], config["target_val"])
    x0, x1 = config["x0"], config["x1"]

    start = time.perf_counter()
    effects = {
        "TV": total_variation(bn, target, config["protected"], x0, x1),
        "TE": total_effect(bn, target, config["protected"], x0, x1),
        "SE": spurious_effect(bn, target, config["protected"], x0),
        "DE": natural_direct_effect(bn, target, config["protected"], x0, x1),
        "IE": natural_indirect_effect(bn, target, config["protected"], x1, x0),
    }
    elapsed = time.perf_counter() - start

    return effects, elapsed


In [ ]:
def build_llm_prompt(config: dict) -> str:
    df = load_filtered(config)
    df, mediators, bin_notes = apply_binning(df, config)

    protected = config["protected"]
    target = config["target_col"]
    target_val = config["target_val"]
    confounders = config["confounders"]

    # Y binario: 1 se target_val, altrimenti 0
    df["_y"] = (df[target] == target_val).astype(int)

    # --- 1. P(Y=y | X) ---
    p_y_given_x = (
        df.groupby(protected, observed=True)["_y"].mean().round(4).reset_index()
        .rename(columns={"_y": "P(Y=y|X)"})
    )

    # --- 2. P(Z) — distribuzione marginale dei confounders ---
    p_z = (
        df.groupby(confounders, observed=True).size().reset_index(name="count")
    )
    p_z["P(Z)"] = (p_z["count"] / len(df)).round(4)
    p_z = p_z.drop(columns="count")

    # --- 3. P(Y=y | X, Z) ---
    p_y_given_xz = (
        df.groupby([protected] + confounders, observed=True)["_y"].mean().round(4).reset_index()
        .rename(columns={"_y": "P(Y=y|X,Z)"})
    )

    # --- 4. P(W | X, Z) ---
    p_w_given_xz = (
        df.groupby([protected] + confounders + mediators, observed=True).size()
        .reset_index(name="count")
    )
    group_totals = p_w_given_xz.groupby([protected] + confounders, observed=True)["count"].transform("sum")
    p_w_given_xz["P(W|X,Z)"] = (p_w_given_xz["count"] / group_totals).round(4)
    p_w_given_xz = p_w_given_xz.drop(columns="count")

    # --- 5. P(Y=y | X, W, Z) ---
    p_y_given_xwz = (
        df.groupby([protected] + mediators + confounders, observed=True)["_y"].mean().round(4).reset_index()
        .rename(columns={"_y": "P(Y=y|X,W,Z)"})
    )

    def to_compact_csv(d: pd.DataFrame) -> str:
        return d.to_csv(index=False)

    binning_note = "\n".join(f"Note: {n}." for n in bin_notes)

    return f"""You are a causal fairness expert. Compute five causal fairness effects
using the Standard Fairness Model (SFM) by Plecko and Bareinboim (2024).

You are given PRE-AGGREGATED CONDITIONAL PROBABILITY TABLES computed from the dataset
(n={len(df)} rows). Use these tables directly — do not assume access to raw data.
{binning_note}

VARIABLE ROLES:
- X (protected): "{protected}", x0="{config['x0']}", x1="{config['x1']}"
- Y (target):    "{target}", target state="{target_val}"
- W (mediators): {mediators}
- Z (confounders): {confounders}

TABLE 1 — P(Y=y | X):
{to_compact_csv(p_y_given_x)}

TABLE 2 — P(Z):
{to_compact_csv(p_z)}

TABLE 3 — P(Y=y | X, Z):
{to_compact_csv(p_y_given_xz)}

TABLE 4 — P(W | X, Z):
{to_compact_csv(p_w_given_xz)}

TABLE 5 — P(Y=y | X, W, Z):
{to_compact_csv(p_y_given_xwz)}

IDENTIFICATION FORMULAE (use these exactly, aggregating over TABLE rows as needed):
- TV = P(Y=y | X=x1) - P(Y=y | X=x0)                    [from TABLE 1]
- TE = sum_z [P(Y=y|x1,z) - P(Y=y|x0,z)] * P(z)          [from TABLE 3, TABLE 2]
- SE = TV - TE
- DE = sum_z,w [P(Y=y|x1,w,z) - P(Y=y|x0,w,z)] * P(w|x0,z) * P(z)   [from TABLE 5, TABLE 4, TABLE 2]
- IE = sum_z,w P(Y=y|x0,w,z) * [P(w|x1,z) - P(w|x0,z)] * P(z)       [from TABLE 5, TABLE 4, TABLE 2]

INSTRUCTIONS:
For DE and IE, the sums run over EVERY combination of z (each row of TABLE 2) and
w (each level of the mediator(s)). For each (z,w) pair you MUST look up the matching
row in TABLE 4 and TABLE 5 by z and w together — do not skip or approximate any term.

Show your work as a short step-by-step calculation for DE and IE (one line per (z,w)
term is fine, or grouped by z), THEN give the final answer.

End your response with a line "FINAL_JSON:" followed by ONLY the JSON object below,
with no markdown formatting:
{{
  "TV": <float>,
  "TE": <float>,
  "SE": <float>,
  "DE": <float>,
  "IE": <float>
}}"""


## 5. Confronto FairMind vs LLM — discrepancies

Stessa funzione del notebook originale: calcola errore assoluto e relativo percentuale tra ground truth (FairMind) e predizione del LLM per ognuno dei 5 effetti causali.

In [ ]:
def compute_discrepancies(ground_truth: dict, llm_effects: dict) -> pd.DataFrame:
    rows = []
    for effect in ["TV", "TE", "SE", "DE", "IE"]:
        gt = ground_truth.get(effect, float("nan"))
        llm_val = float(llm_effects.get(effect, float("nan")))
        abs_err = abs(gt - llm_val)
        rel_err = abs_err / abs(gt) if abs(gt) > 1e-9 else float("nan")
        rows.append({
            "effect": effect,
            "fairmind": round(gt, 6),
            "llm": round(llm_val, 6),
            "abs_error": round(abs_err, 6),
            "rel_error_%": round(rel_err * 100, 2) if not pd.isna(rel_err) else float("nan"),
        })
    return pd.DataFrame(rows)


def save_results(config, ground_truth, llm_effects, discrepancies, usage, fairmind_time, llm_time):
    os.makedirs("benchmark_results/multi_dataset", exist_ok=True)
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    fname = f"benchmark_results/multi_dataset/{config['dataset_name']}_{ts}.json"

    out = {
        "dataset": config["dataset_name"],
        "config": {k: v for k, v in config.items() if k != "csv_path"},
        "fairmind": ground_truth,
        "llm": llm_effects,
        "discrepancies": discrepancies.to_dict(orient="records"),
        "token_usage": usage,
        "timing": {
            "fairmind_seconds": round(fairmind_time, 4),
            "llm_seconds": round(llm_time, 4),
        },
    }
    with open(fname, "w") as f:
        json.dump(out, f, indent=2)
    return fname


## 6. Loop del benchmark su tutti i dataset

Esegue in sequenza FairMind + LLM per ogni dataset in `CONFIGS`. Un errore su un singolo dataset (es. parsing della risposta LLM, colonna mancante) viene loggato e non interrompe il loop sugli altri dataset.

In [ ]:
all_discrepancies = []  # righe per la tabella riassuntiva finale
failed = []

for config in CONFIGS:
    name = config["dataset_name"]
    print(f"\n=== {name} ===")
    try:
        ground_truth, fairmind_time = run_fairmind(config)
        print(f"FairMind — elapsed: {fairmind_time:.4f}s — " +
              ", ".join(f"{k}={v:.4f}" for k, v in ground_truth.items()))

        prompt = build_llm_prompt(config)
        llm_effects, llm_usage, llm_time = call_llm(prompt, max_tokens=8192)
        print(f"LLM — elapsed: {llm_time:.4f}s — token totali: {llm_usage['total_tokens']}")

        discrepancies = compute_discrepancies(ground_truth, llm_effects)
        fname = save_results(config, ground_truth, llm_effects, discrepancies, llm_usage, fairmind_time, llm_time)
        print(f"Saved: {fname}")

        discrepancies.insert(0, "dataset", name)
        all_discrepancies.append(discrepancies)
    except Exception as e:
        print(f"ERRORE su {name}: {e}")
        failed.append((name, str(e)))

if failed:
    print(f"\n{len(failed)} dataset falliti: {[n for n, _ in failed]}")


## 7. Tabella riassuntiva multi-dataset

In [ ]:
summary_df = pd.concat(all_discrepancies, ignore_index=True) if all_discrepancies else pd.DataFrame()
print(summary_df.to_string(index=False))


In [ ]:
# Errore medio per dataset (su tutti i 5 effetti)
if not summary_df.empty:
    per_dataset = (
        summary_df.groupby("dataset")[["abs_error", "rel_error_%"]]
        .mean()
        .round(4)
        .sort_values("abs_error", ascending=False)
    )
    print(per_dataset.to_string())


## 8. Salvataggio riassunto aggregato

In [ ]:
if not summary_df.empty:
    os.makedirs("benchmark_results/multi_dataset", exist_ok=True)
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    out_csv = f"benchmark_results/multi_dataset/multi_dataset_summary_{ts}.csv"
    summary_df.to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}")
